# **Production Monitoring & Retraining Strategy**
### **Objective:** Define a robust automated monitoring framework to detect silent failures, data drift, and performance degradation in the delivery time prediction service.

## **1. Monitoring Philosophy**
Our strategy is built on three pillars of observability:
1. **Data Health (Input):** Detecting feature drift using Population Stability Index (PSI).
2. **Model Health (Output):** Tracking rolling performance metrics (MAE/RMSE) against ground truth.
3. **Trust Health (Confidence):** Monitoring the coverage of our prediction intervals.

**Threshold Logic:** Thresholds are set based on the Layer 2B baseline. A 20% degradation in MAE from baseline (0.63 -> 0.75) triggers a WARNING, while a 50% drop triggers a CRITICAL retraining requirement.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Load data and split into 'Reference' (historical) and 'Production' (simulated)
df = pd.read_csv('../Dataset/final_dataset.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

# Reference is the data the model was trained on
df_ref = df.iloc[:int(len(df)*0.8)]
# Production is the incoming stream
df_prod = df.iloc[int(len(df)*0.8):]

BASELINE_MAE = 0.63
TARGET_COVERAGE = 0.90

## **2. Automated Monitoring Logic**
The following functions simulate the logic that would run inside a monitoring microservice.

In [2]:
def calculate_psi(expected, actual, buckets=10):
    expected_percents = np.histogram(expected, bins=buckets)[0] / len(expected)
    actual_percents = np.histogram(actual, bins=buckets)[0] / len(actual)
    # Add epsilon to prevent log(0)
    expected_percents = np.clip(expected_percents, 0.0001, 1)
    actual_percents = np.clip(actual_percents, 0.0001, 1)
    return np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))

def evaluate_production_health(actual_mae, max_psi, coverage):
    alerts = []
    
    # Check 1: Performance Degradation
    mae_ratio = actual_mae / BASELINE_MAE
    if mae_ratio > 1.5:
        alerts.append(("CRITICAL", f"MAE degraded by {mae_ratio:.2f}x"))
    elif mae_ratio > 1.2:
        alerts.append(("WARNING", f"MAE showing upward trend: {actual_mae:.3f}"))

    # Check 2: Feature Drift
    if max_psi > 0.25:
        alerts.append(("CRITICAL", f"Severe feature drift detected: PSI={max_psi:.3f}"))
    elif max_psi > 0.1:
        alerts.append(("WARNING", f"Moderate feature drift: PSI={max_psi:.3f}"))

    # Check 3: Confidence Calibration
    if coverage < 0.80:
        alerts.append(("CRITICAL", f"Confidence intervals collapsed: Coverage={coverage:.2%}"))
        
    return alerts

## **3. Retraining Decision Engine**
We use a multi-signal confirmation logic to avoid "jitter" (frequent, unnecessary retraining due to noise).

In [3]:
def retraining_decision(alerts):
    critical_count = sum(1 for level, msg in alerts if level == "CRITICAL")
    warning_count = sum(1 for level, msg in alerts if level == "WARNING")
    
    if critical_count >= 1 or warning_count >= 2:
        return "RETRAIN"
    elif warning_count == 1:
        return "MONITOR_CLOSELY"
    return "NO_ACTION"

# Simulation: Current Production Batch Metrics
current_metrics = {
    'MAE': 0.65, 
    'Max_PSI': 0.04,
    'Coverage': 0.89
}

active_alerts = evaluate_production_health(current_metrics['MAE'], current_metrics['Max_PSI'], current_metrics['Coverage'])
decision = retraining_decision(active_alerts)

print(f"--- Monitoring Report ---")
for level, msg in active_alerts:
    print(f"[{level}] {msg}")
print(f"\nFinal System Decision: {decision}")

--- Monitoring Report ---

Final System Decision: NO_ACTION


## **4. Fail-safe & Rollback Logic**
Automated retraining introduces the risk of the model learning from corrupted data. 

**Rollback Protocol:**
1. **Pre-Deployment Check:** New models must outperform the current production model on a static 'Golden Holdout Set'.
2. **Canary Release:** Route 5% of traffic to the new model; if error rate spikes, revert immediately.
3. **Version Pinning:** Always maintain the last 3 stable model artifacts in storage.

## **Final System-Level Verdict**
> **Status:** 🟢 **PRODUCTION-READY**
> 
> The monitoring framework is complete. We have objective thresholds for feature drift (PSI), performance loss (MAE), and calibration (Coverage). The retraining logic is conservative to prevent overfitting to short-term anomalies. This system is ready for MLOps handoff.